# Improving Conversion in E-commerce Funnel

## Objective

The goal of this analysis is to understand user behavior across the purchase funnel and identify key bottlenecks impacting conversion. Additionally, we aim to quantify the potential business impact of improving these bottlenecks.

---

In [1]:
import pandas as pd 
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("events.csv")

## Data Understanding

The dataset contains user interaction data from an e-commerce platform, capturing different stages of the purchase journey.
 


### Key Columns

- **event_time**: Timestamp of the user action  
- **event_type**: Type of interaction (`view`, `cart`, `purchase`)  
- **product_id**: Unique identifier for the product  
- **category_id / category_code**: Product category hierarchy  
- **brand**: Brand associated with the product  
- **price**: Product price  
- **user_id**: Unique identifier for the user  
- **user_session**: Identifier for a browsing session 

In [3]:
df.head(10)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,NaN,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,NaN,NaN,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,NaN,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ
5,2020-09-24 11:57:59 UTC,view,664325,2144415951611757447,construction.tools.saw,carver,52.33,1515915625519388062,vnkdP81DDW
6,2020-09-24 11:58:23 UTC,view,3791349,2144415935086199225,computers.desktop,NaN,215.41,1515915625519388877,J1t6sIYXiV
7,2020-09-24 11:58:24 UTC,view,716611,2144415923694469257,computers.network.router,d-link,53.14,1515915625519388882,kVBeYDPcBw
8,2020-09-24 11:58:25 UTC,view,657859,2144415939431498289,NaN,NaN,34.17,1515915625519320570,HEl15U7JVy
9,2020-09-24 11:58:31 UTC,view,716611,2144415923694469257,computers.network.router,d-link,53.14,1515915625519388929,F3VB9LYp39


In [4]:
df.tail(10)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
885119,2021-02-28 23:51:08 UTC,view,733804,2144415958968565847,appliances.kitchen.juicer,NaN,35.00,1515915625611023253,ds44R7DkhX
885120,2021-02-28 23:51:25 UTC,view,622796,2144415922738167921,computers.components.cdrw,asus,147.38,1515915625572947504,LJ4H6CRcME
885121,2021-02-28 23:51:25 UTC,view,622796,2144415922738167921,computers.components.cdrw,asus,147.38,1515915625572947504,SqlXaC3Wrw
885122,2021-02-28 23:53:13 UTC,view,4079420,2144415922427789416,computers.components.videocards,msi,449.51,1515915625611023581,zrl0oKrysT
885123,2021-02-28 23:54:18 UTC,view,3829355,2144415922528452715,electronics.telephone,NaN,32.22,1515915625611023671,wZb7gP1zgN
885124,2021-02-28 23:55:01 UTC,view,953226,2144415927553229037,NaN,NaN,219.94,1515915625611023730,FRLqIttxKU
885125,2021-02-28 23:58:05 UTC,view,1715907,2144415927049912542,electronics.video.tv,starwind,80.03,1515915625611024014,g6WqPf50Ma
885126,2021-02-28 23:58:09 UTC,view,4170534,2144415939364389423,electronics.clocks,amazfit,64.92,1515915625611024020,xNIJBqZdkd
885127,2021-02-28 23:58:14 UTC,view,888273,2144415921932861531,electronics.telephone,NaN,10.16,1515915625611024030,9pCbKMIcSx
885128,2021-02-28 23:59:09 UTC,view,743182,2144415935631458761,construction.tools.soldering,kada,65.08,1515915625556087775,BejOXRngEW


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885129 entries, 0 to 885128
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     885129 non-null  object 
 1   event_type     885129 non-null  object 
 2   product_id     885129 non-null  int64  
 3   category_id    885129 non-null  int64  
 4   category_code  648910 non-null  object 
 5   brand          672765 non-null  object 
 6   price          885129 non-null  float64
 7   user_id        885129 non-null  int64  
 8   user_session   884964 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 60.8+ MB


In [6]:
df.describe()

,product_id,category_id,price,user_id
count,8.851290e+05,8.851290e+05,885129.000000,8.851290e+05
mean,1.906621e+06,2.144423e+18,146.328713,1.515916e+18
std,1.458708e+06,6.165105e+14,296.807683,3.747287e+07
min,1.020000e+02,2.144416e+18,0.220000,1.515916e+18
25%,6.988030e+05,2.144416e+18,26.460000,1.515916e+18
50%,1.452883e+06,2.144416e+18,65.710000,1.515916e+18
75%,3.721194e+06,2.144416e+18,190.490000,1.515916e+18
max,4.183880e+06,2.227847e+18,64771.060000,1.515916e+18


In [7]:
df.isna().sum()

event_time            0
event_type            0
product_id            0
category_id           0
category_code    236219
brand            212364
price                 0
user_id               0
user_session        165
dtype: int64

In [8]:
df[df['user_session'].isna()]['event_type'].value_counts()

event_type
view    159
cart      6
Name: count, dtype: int64

In [9]:
print("Total rows:", len(df))
print("Date range:", df['event_time'].min(), "to", df['event_time'].max())
print("Unique users:", df['user_id'].nunique())
print("Unique sessions:", df['user_session'].nunique())

print("\nMissing %:\n",df.isna().mean() * 100)

print("\nDuplicate rows:", df.duplicated().sum())

Total rows: 885129
Date range: 2020-09-24 11:57:06 UTC to 2021-02-28 23:59:09 UTC
Unique users: 407283
Unique sessions: 490398

Missing %:
 event_time        0.000000
event_type        0.000000
product_id        0.000000
category_id       0.000000
category_code    26.687522
brand            23.992435
price             0.000000
user_id           0.000000
user_session      0.018641
dtype: float64

Duplicate rows: 655


### Data Overview & Sanity Checks

- Total records: ~885K  
- Time range: Sep 2020 – Feb 2021 (~5 months)  
- Unique users: ~407K  
- Unique sessions: ~490K  

### Data Quality Checks

- No missing values across key fields  
- Duplicate rows: ~650 (<0.1% of data, negligible impact)  
- Session-level granularity is sufficient for funnel analysis  


**Insight:**  
- The dataset is large, clean, and reliable for session-based funnel analysis, with minimal data quality concerns.
-  Average sessions per user: ~1.2 (indicates mostly short interaction cycles)

---

## Data Preparation

The dataset was prepared to ensure accurate session-level analysis and consistent categorical representation.


### 1. Handling Missing Values

- Rows with missing `user_session` were removed, as session-level tracking is essential for funnel construction.
- Missing values in `category_code` and `brand` were replaced with `"unknown"` to retain data while explicitly marking incomplete information.

---

In [10]:
df = df.dropna(subset='user_session')

df['category_code'] = df['category_code'].fillna('unknown')
df['brand'] = df['brand'].fillna('unknown')

df.isna().sum()

event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     0
dtype: int64


### 2. Feature Engineering

- A new column `category` was created by extracting the top-level category from `category_code` - this allows aggregation at a meaningful level while avoiding excessive granularity.

In [11]:
df['category'] = df['category_code'].str.split('.').str[0]

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,unknown,31.90,1515915625519388267,LJuJVLEjPT,electronics
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY,computers
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,unknown,unknown,9.81,1515915625513238515,4TMArHtXQy,unknown
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08,computers
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,unknown,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ,unknown


## Analytical Approach

Given the prepared dataset, the analysis will:

- Construct a **session-based funnel** to measure user progression across stages  
- Segment behavior across **categories, sub-categories, and brands**  
- Estimate the potential **business impact of improving conversion rates** 
- Identify key drivers of **add-to-cart conversion**, the primary bottleneck in the funnel  

---

## Funnel Analysis

A session-based funnel was constructed to track user progression.


In [12]:
funnel = (
    df.groupby(['user_session', 'event_type'])
      .size()
      .unstack(fill_value=0)
)

view_users = (funnel['view'] > 0).sum()
cart_users = (funnel['cart'] > 0).sum()
purchase_users = (funnel['purchase'] > 0).sum()

cart_rate = cart_users / view_users
purchase_rate = purchase_users / cart_users
overall_rate = purchase_users/view_users

print(f"Cart Rate: {cart_rate*100}")
print(f"Purchase Rate: {purchase_rate*100}")
print(f"Overall Rate: {overall_rate*100}")



stages = ['View', 'Cart', 'Purchase']
values = [view_users, cart_users, purchase_users]

labels = [
    f"{view_users:,}",
    f"{cart_users:,} ({cart_rate*100:.1f}%)",
    f"{purchase_users:,} ({purchase_rate*100:.1f}%)"
]

fig = go.Figure(go.Funnel(
    y=stages,
    x=values,
    text=labels,
    textposition="outside",   
    marker=dict(
        color=["#636EFA", "#00CC96", "#EF553B"]
    )
))

fig.update_layout(
    title="User Funnel Conversion",
    width=550,
    height=420,
    template="plotly_white",
    margin=dict(l=40, r=40, t=60, b=60)
)
fig.data[0].textposition = ["inside", "outside", "outside"]
fig.add_annotation(
    text=f"~{(1-cart_rate)*100:.1f}% drop at View → Cart",
    xref="paper", yref="paper",
    x=0.5, y=-0.15,
    showarrow=False
)
fig.update_traces(opacity=0.9)
fig.show()

Cart Rate: 8.450733065771153
Purchase Rate: 58.98715774170099
Overall Rate: 4.984847243836514


Converstion Rates :
- View → Cart: ~8.5%
- Cart → Purchase: ~59%
- Overall: ~5%

**Insight:** Major drop at add-to-cart stage → primary bottleneck. Purchase stage is not the problem.

---

## Monthly Trend Analysis

To evaluate whether conversion behavior changed over time, monthly trends were analyzed.

In [13]:
# ensure datetime
df['event_time'] = pd.to_datetime(df['event_time'])

# create month column
df['month'] = df['event_time'].dt.to_period('M').astype(str)

# session-level funnel
monthly = (
    df.groupby(['month', 'user_session', 'event_type'])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

# binary funnel
monthly['view'] = (monthly['view'] > 0).astype(int)
monthly['cart'] = (monthly['cart'] > 0).astype(int)
monthly['purchase'] = (monthly['purchase'] > 0).astype(int)

# aggregate
monthly_summary = monthly.groupby('month').agg({
    'view': 'sum',
    'cart': 'sum',
    'purchase': 'sum'
})

# rates
monthly_summary['cart_rate'] = monthly_summary['cart'] / monthly_summary['view']
monthly_summary['purchase_rate'] = monthly_summary['purchase'] / monthly_summary['cart']

monthly_summary = monthly_summary.reset_index()

# plot
fig = px.line(
    monthly_summary,
    x='month',
    y=['cart_rate', 'purchase_rate'],
    markers=True,
    title='Monthly Conversion Trends'
)

fig.update_layout(
    width=700,
    height=400,
    template='plotly_white',
    yaxis_title='Conversion Rate',
    xaxis_title=''
)

fig.show()

**Insight:** Cart conversion shows a slight upward trend over time, while purchase conversion remains largely stable with minor fluctuations. These variations are not substantial enough to indicate a consistent trend, reinforcing that the primary drivers of conversion are structural rather than temporal.

---

## Category Distribution & Behavior

- Top categories: Computers (~36%), Unknown (~27%), Electronics (~19%)

In [14]:
df['category'].value_counts(normalize=True)*100


category
computers       35.809931
unknown         26.687187
electronics     19.330391
stationery       4.856130
appliances       4.641997
auto             4.003666
construction     3.509408
furniture        0.380241
country_yard     0.355043
accessories      0.235038
medicine         0.080568
kids             0.042036
jewelry          0.028815
sport            0.026216
apparel          0.013334
Name: proportion, dtype: float64

In [15]:
print(f"\nComputers:\n{df[df['category'] == 'computers']['event_type'].value_counts(normalize=True)}\n")

print(f"Unknown:\n{df[df['category'] == 'unknown']['event_type'].value_counts(normalize=True)}\n")

print(f"Electronics:\n{df[df['category'] == 'electronics']['event_type'].value_counts(normalize=True)}")


Computers:
event_type
view        0.861624
cart        0.085142
purchase    0.053234
Name: proportion, dtype: float64

Unknown:
event_type
view        0.923886
cart        0.044070
purchase    0.032044
Name: proportion, dtype: float64

Electronics:
event_type
view        0.907487
cart        0.052950
purchase    0.039563
Name: proportion, dtype: float64


**Insight:** Computers show stronger conversion behavior, while Unknown and Electronics are more view-heavy (lower intent).

---

### Category-Level Insights

Conversion rates were compared across categories.


In [16]:
funnel_cat = (
    df.groupby(['category', 'user_session', 'event_type'])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

funnel_cat['view'] = (funnel_cat['view'] > 0).astype(int)
funnel_cat['cart'] = (funnel_cat['cart'] > 0).astype(int)
funnel_cat['purchase'] = (funnel_cat['purchase'] > 0).astype(int)

summary = funnel_cat.groupby('category').agg({
    'view': 'sum',
    'cart': 'sum',
    'purchase': 'sum'
})

summary['cart_rate'] = summary['cart'] / summary['view']
summary['purchase_rate'] = summary['purchase'] / summary['cart']
summary['overall_rate'] = summary['purchase'] / summary['view']

# filter noise
summary = summary[(summary['view'] > 1000) & (summary['purchase'] > 50)]
#----------------------------------------------------------------------------------------------------------------------#

# prepare data for graph
plot_df = summary.copy().reset_index()

plot_df['cart_rate_pct'] = plot_df['cart_rate'] * 100
plot_df['purchase_rate_pct'] = plot_df['purchase_rate'] * 100

# sort by cart rate (important for readability)
plot_df = plot_df.sort_values('cart_rate_pct', ascending=False)

# create subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Cart Rate by Category", "Purchase Rate by Category")
)

# --- Cart Rate ---
fig.add_trace(
    go.Bar(
        x=plot_df['category'],
        y=plot_df['cart_rate_pct'],
        text=plot_df['cart_rate_pct'].round(1),
        textposition='outside',
        name='Cart Rate',
        marker_color='#636EFA'
    ),
    row=1, col=1
)

# --- Purchase Rate ---
fig.add_trace(
    go.Bar(
        x=plot_df['category'],
        y=plot_df['purchase_rate_pct'],
        text=plot_df['purchase_rate_pct'].round(1),
        textposition='outside',
        name='Purchase Rate',
        marker_color='#00CC96'
    ),
    row=1, col=2
)

# layout
fig.update_layout(
    width=900,
    height=420,
    template='plotly_white',
    showlegend=False,
    margin=dict(l=40, r=40, t=60, b=80)
)

# axis formatting
fig.update_xaxes(tickangle=-30)

fig.update_yaxes(title_text="Cart Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Purchase Rate (%)", row=1, col=2)

# start from 0
fig.update_yaxes(range=[0, plot_df['cart_rate_pct'].max() * 1.2], row=1, col=1)
fig.update_yaxes(range=[0, plot_df['purchase_rate_pct'].max() * 1.2], row=1, col=2)

fig.show()
summary.sort_values('overall_rate', ascending=False)

event_type,view,cart,purchase,cart_rate,purchase_rate,overall_rate
category,,,,,,
computers,148481,19588,10964,0.131923,0.559730,0.073841
stationery,26926,2492,1748,0.092550,0.701445,0.064919
electronics,103281,7614,4679,0.073721,0.614526,0.045304
furniture,2146,139,85,0.064772,0.611511,0.039609
auto,19698,1257,738,0.063814,0.587112,0.037466
construction,17602,1154,655,0.065561,0.567591,0.037212
unknown,149524,8785,5284,0.058753,0.601480,0.035339
appliances,25303,1010,617,0.039916,0.610891,0.024384


- Cart rate varies widely (~4% → ~13%)
- Purchase rate relatively stable (~55–65%)

**Insight:** Performance driven by cart rate, not purchase rate.

---

## Business Impact Estimation

Having identified the key drivers, we now estimate the potential business impact of improving conversion.

To estimate potential gains, cart rates were improved toward the top-performing category (benchmark) by partially closing the gap.

- Improvement applied: 50% of gap to benchmark  
- Expected purchases recalculated using improved cart rates  


In [17]:
# 1) Benchmark (top performer)
benchmark_cart_rate = summary['cart_rate'].max()

# improvement
improvement_factor = 0.5
gap = (benchmark_cart_rate - summary['cart_rate']).clip(lower=0)

summary['improved_cart_rate'] = summary['cart_rate'] + improvement_factor * gap

# expected funnel
summary['expected_cart'] = (summary['view'] * summary['improved_cart_rate']).astype(int)
summary['expected_purchase'] = (summary['expected_cart'] * summary['purchase_rate']).astype(int)

# uplift
summary['uplift'] = summary['expected_purchase'] - summary['purchase']

# sort
summary.sort_values('uplift', ascending=False)

event_type,view,cart,purchase,cart_rate,purchase_rate,overall_rate,improved_cart_rate,expected_cart,expected_purchase,uplift
category,,,,,,,,,,
unknown,149524,8785,5284,0.058753,0.601480,0.035339,0.095338,14255,8574,3290
electronics,103281,7614,4679,0.073721,0.614526,0.045304,0.102822,10619,6525,1846
appliances,25303,1010,617,0.039916,0.610891,0.024384,0.085919,2174,1328,711
auto,19698,1257,738,0.063814,0.587112,0.037466,0.097868,1927,1131,393
stationery,26926,2492,1748,0.092550,0.701445,0.064919,0.112236,3022,2119,371
construction,17602,1154,655,0.065561,0.567591,0.037212,0.098742,1738,986,331
furniture,2146,139,85,0.064772,0.611511,0.039609,0.098347,211,129,44
computers,148481,19588,10964,0.131923,0.559730,0.073841,0.131923,19588,10964,0



**Insight:** Even partial improvements in cart rate can lead to meaningful increases in purchases, especially in high-volume categories.

---

### Revenue Impact

Average purchase price was used to estimate the revenue impact of improved conversion.

- Revenue uplift calculated using expected additional purchases  
- Total uplift quantified across all categories  

- Estimated revenue uplift: ~15.3%

In [18]:
# --- Revenue calculation ---
purchase_df = df[df['event_type'] == 'purchase']

avg_price = purchase_df.groupby('category')['price'].mean()

# assign directly
summary['avg_price'] = avg_price

summary['current_revenue'] = (summary['purchase'] * summary['avg_price']).astype(int)
summary['revenue_uplift'] = (summary['uplift'] * summary['avg_price']).astype(int)

# --- Total impact ---
total_current_revenue = summary['current_revenue'].sum()
total_revenue_uplift = summary['revenue_uplift'].sum()

uplift_pct = (total_revenue_uplift / total_current_revenue) * 100

print(f"Total Revenue Uplift: {uplift_pct:.2f}%")

# --- Final prioritization ---
summary = summary.sort_values(by='revenue_uplift', ascending=False)

summary


Total Revenue Uplift: 15.24%


event_type,view,cart,purchase,cart_rate,purchase_rate,overall_rate,improved_cart_rate,expected_cart,expected_purchase,uplift,avg_price,current_revenue,revenue_uplift
category,,,,,,,,,,,,,
unknown,149524,8785,5284,0.058753,0.601480,0.035339,0.095338,14255,8574,3290,65.266031,344865,214725
electronics,103281,7614,4679,0.073721,0.614526,0.045304,0.102822,10619,6525,1846,66.540864,311344,122834
appliances,25303,1010,617,0.039916,0.610891,0.024384,0.085919,2174,1328,711,116.482290,71869,82818
auto,19698,1257,738,0.063814,0.587112,0.037466,0.097868,1927,1131,393,107.653690,79448,42307
construction,17602,1154,655,0.065561,0.567591,0.037212,0.098742,1738,986,331,108.791191,71258,36009
stationery,26926,2492,1748,0.092550,0.701445,0.064919,0.112236,3022,2119,371,37.571216,65674,13938
furniture,2146,139,85,0.064772,0.611511,0.039609,0.098347,211,129,44,18.181043,1545,799
computers,148481,19588,10964,0.131923,0.559730,0.073841,0.131923,19588,10964,0,221.033524,2423411,0


**Insight:** Conversion improvements translate directly into significant revenue gains, with the largest impact coming from high-volume categories.

---

#### Sensitivity Analysis

To account for uncertainty in improvement assumptions, multiple scenarios were tested by varying the proportion of the gap closed toward the benchmark.

- Conservative: 30% gap closure  
- Moderate: 50% gap closure  
- Aggressive: 100% gap closure  

Revenue uplift was recalculated under each scenario.

In [19]:
# Define improvement scenarios (fraction of gap closed toward benchmark)
scenarios = {
    'conservative': 0.3,   # close 30% of gap
    'moderate': 0.5,       # close 50% of gap
    'aggressive': 1.0      # fully reach benchmark
}

results = {}

# Use best-performing category as benchmark
benchmark_cart_rate = summary['cart_rate'].max()

for name, factor in scenarios.items():
    
    temp = summary.copy()  
    
    # Calculate gap to benchmark
    gap = benchmark_cart_rate - temp['cart_rate']
    
    # Apply improvement (partial gap closure)
    temp['improved_cart_rate'] = temp['cart_rate'] + factor * gap
    
    # Ensure rates do not exceed benchmark
    temp['improved_cart_rate'] = temp['improved_cart_rate'].clip(upper=benchmark_cart_rate)
    
    # Recompute expected funnel with improved rates
    temp['expected_cart'] = temp['view'] * temp['improved_cart_rate']
    temp['expected_purchase'] = temp['expected_cart'] * temp['purchase_rate']
    
    # Incremental purchases
    temp['uplift'] = temp['expected_purchase'] - temp['purchase']
    
    # Revenue impact from additional purchases
    temp['revenue_uplift'] = temp['uplift'] * temp['avg_price']
    
    # Total revenue before improvement
    total_current_revenue = (temp['purchase'] * temp['avg_price']).sum()
    
    # Total revenue after improvement
    total_new_revenue = total_current_revenue + temp['revenue_uplift'].sum()
    
    # % uplift in revenue
    uplift_pct = (total_new_revenue - total_current_revenue) / total_current_revenue * 100
    
    # Store result for scenario
    results[name] = uplift_pct


import plotly.graph_objects as go

# dataframe
df_plot = pd.DataFrame({
    'Scenario': list(results.keys()),
    'Revenue Uplift (%)': list(results.values())
})

# order
order = ['conservative', 'moderate', 'aggressive']
df_plot['Scenario'] = pd.Categorical(df_plot['Scenario'], categories=order, ordered=True)
df_plot = df_plot.sort_values('Scenario')

# formatted labels
labels = [f"{x:.1f}%" for x in df_plot['Revenue Uplift (%)']]

# plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot['Scenario'],
    y=df_plot['Revenue Uplift (%)'],
    mode='lines+markers+text',   
    text=labels,
    textposition='top center'
))

# layout
fig.update_layout(
    width=500,
    height=400,
    template='plotly_white',
    showlegend=False,
    title='Estimated Revenue Uplift by Scenario',
    xaxis_title='',
    yaxis_title='Revenue Uplift (%)',
    margin=dict(l=40, r=40, t=60, b=40)
)

# axis from 0
fig.update_yaxes(range=[0, df_plot['Revenue Uplift (%)'].max() * 1.1])

fig.show()


**Insight:** Even under conservative assumptions, improving cart rates yields meaningful revenue gains, while aggressive scenarios highlight the upper bound of potential impact.

---

## Second Level Analysis  
To better understand the drivers behind these differences, a deeper analysis was conducted at the sub-category and brand levels.

### Brand-Level (Electronics)

To understand variation within electronics, conversion behavior was analyzed at the brand level.

- Cart rates vary significantly across brands  
- Low-volume brands were filtered out to reduce noise  



In [20]:
# Filter only electronics category for second-level analysis
elec = df[df['category'] == 'electronics']

# Build session-level funnel at brand level
brand_funnel = (
    elec.groupby(['brand', 'user_session', 'event_type'])  # group by brand + session + event
        .size()                                            # count events
        .unstack(fill_value=0)                             # pivot event_type into columns
        .reset_index()
)

# Convert to binary funnel (1 = event occurred in session)
brand_funnel['view'] = (brand_funnel['view'] > 0).astype(int)
brand_funnel['cart'] = (brand_funnel['cart'] > 0).astype(int)
brand_funnel['purchase'] = (brand_funnel['purchase'] > 0).astype(int)

# Aggregate to brand level
brand_summary = brand_funnel.groupby('brand').agg({
    'view': 'sum',
    'cart': 'sum',
    'purchase': 'sum'
})

# Calculate cart conversion rate
brand_summary['cart_rate'] = brand_summary['cart'] / brand_summary['view']

# Remove low-volume brands to avoid noisy estimates
brand_summary = brand_summary[brand_summary['view'] > 500]

# Sort by scale and performance
brand_summary.sort_values(['view', 'cart_rate'], ascending=[False, True])

event_type,view,cart,purchase,cart_rate
brand,,,,
unknown,28897,3133,2084,0.108420
samsung,11121,579,359,0.052064
sirius,6413,773,528,0.120536
bbk,3506,176,85,0.050200
lg,3325,29,13,0.008722
xiaomi,3289,214,86,0.065065
edifier,2776,247,149,0.088977
digma,2236,170,87,0.076029
starwind,1919,170,82,0.088588


**Insight:** High-volume brands such as Samsung and Apple show significantly lower cart rates compared to lower-priced or lesser-known brands, indicating stronger comparison behavior and higher hesitation at premium price points.

---

### Brand × Price Analysis

To further explain brand-level differences, average purchase price was incorporated.

In [21]:
# --- Filter electronics category ---
elec = df[df['category'] == 'electronics'].copy()

# --- Build session-level funnel at brand level ---
brand_funnel = (
    elec.groupby(['brand', 'user_session', 'event_type'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
)

# --- Convert to binary funnel (per session) ---
brand_funnel['view'] = (brand_funnel['view'] > 0).astype(int)
brand_funnel['cart'] = (brand_funnel['cart'] > 0).astype(int)
brand_funnel['purchase'] = (brand_funnel['purchase'] > 0).astype(int)

# --- Aggregate to brand level ---
brand_summary = brand_funnel.groupby('brand').agg({
    'view': 'sum',
    'cart': 'sum',
    'purchase': 'sum'
})

# --- Conversion rate ---
brand_summary['cart_rate'] = brand_summary['cart'] / brand_summary['view']

# --- Remove low-volume brands (noise control) ---
brand_summary = brand_summary[brand_summary['view'] > 2000]

# --- Add average purchase price ---
purchase_df = df[df['event_type'] == 'purchase']

avg_price = purchase_df.groupby('brand')['price'].mean()

brand_summary['avg_price'] = avg_price

# --- Final sorting ---
brand_summary.sort_values('cart_rate')

event_type,view,cart,purchase,cart_rate,avg_price
brand,,,,,
lg,3325,29,13,0.008722,217.752281
bbk,3506,176,85,0.050200,90.608409
samsung,11121,579,359,0.052064,72.063984
xiaomi,3289,214,86,0.065065,174.172606
digma,2236,170,87,0.076029,88.373437
edifier,2776,247,149,0.088977,144.681912
unknown,28897,3133,2084,0.108420,62.596568
sirius,6413,773,528,0.120536,23.050606


**Insight:** Higher-priced brands such as Sony (~$313) and LG (~$218) show significantly lower cart rates (~0.7%) compared to lower-priced brands like Samsung (~$72, ~4.3%) and Sirius (~$23, ~10%). This suggests that users are more hesitant when interacting with higher-priced products, likely due to increased evaluation and comparison within similar options.

---

### Price-Level Analysis

To evaluate the impact of pricing on conversion, products were grouped into price buckets and analyzed at the session level.

- Cart rates were compared across price segments  
- Sessions were used to avoid bias from repeated user actions  

*Note: Price bucket is assigned at the session level using the first observed product price, which serves as an approximation when multiple products are viewed within a session.*

In [22]:
# Reduce dataset to relevant columns for efficiency
df_small = df[['price', 'user_session', 'event_type']].copy()

# Create price buckets (quantiles) to segment products by price
df_small['price_bucket'] = pd.qcut(df_small['price'], 5, duplicates='drop')

# Build session-level event matrix
session_events = (
    df_small.groupby(['user_session', 'event_type'])  # group by session + event
            .size()                                   # count events
            .unstack(fill_value=0)                    # pivot event types into columns
)

# Convert to binary funnel (1 = event occurred in session)
session_events = (session_events > 0).astype(int)

# Assign one price bucket per session (approximate using first occurrence)
session_price = df_small.groupby('user_session')['price_bucket'].first()

# Merge price bucket into session-level data
session_events = session_events.join(session_price)

# Aggregate at price bucket level
price_summary = session_events.groupby('price_bucket').sum()

# Calculate cart conversion rate
price_summary['cart_rate'] = price_summary['cart'] / price_summary['view']

price_summary

,cart,purchase,view,cart_rate
price_bucket,,,,
"(0.219, 21.33]",7401,4714,112056,0.066047
"(21.33, 46.67]",8184,5489,105021,0.077927
"(46.67, 96.49]",6874,4080,97055,0.070826
"(96.49, 226.84]",8605,4883,89861,0.095759
"(226.84, 64771.06]",10206,5178,84367,0.120971


**Insight:** Higher price buckets show stronger cart rates (~12% vs ~6% in lower buckets), suggesting that users interacting with higher-priced products are typically more intentional, though this effect reverses when comparing within similar product categories.

---

### Sub-category Analysis (Electronics)

To further investigate variation within electronics, performance was analyzed at the sub-category level.

- Cart rates vary significantly across sub-categories  
- Low-volume segments were filtered out to reduce noise  

In [23]:
# Filter electronics category for deeper analysis
elec = df[df['category'] == 'electronics'].copy()

# Extract sub-category from category_code (second level)
elec['sub_category'] = elec['category_code'].str.split('.').str[1]

# Aggregate events at sub-category level
sub_summary = (
    elec.groupby(['sub_category', 'event_type'])  # group by sub-category + event
        .size()                                   # count events
        .unstack(fill_value=0)                    # pivot event types into columns
)

# Calculate cart conversion rate
sub_summary['cart_rate'] = sub_summary['cart'] / sub_summary['view']

# Remove low-volume sub-categories to avoid noisy estimates
sub_summary = sub_summary[sub_summary['view'] > 1000]

# Sort by cart rate to identify best and worst performers
sub_summary.sort_values('cart_rate')

event_type,cart,purchase,view,cart_rate
sub_category,,,,
clocks,62,35,3441,0.018018
video,859,545,22631,0.037957
audio,1561,1161,34545,0.045187
camera,98,79,1657,0.059143
tablet,1084,819,17473,0.062039
telephone,5385,4119,74839,0.071954


**Insight:** Sub-categories such as telephones exhibit higher cart rates (~7%), while others like clocks and video products show significantly lower conversion (~2–4%), indicating differences in user intent and purchase urgency.

---

### Temporal Check

To validate whether conversion behavior is influenced by time, cart rates were analyzed across hours of the day.

In [ ]:
# ensure datetime
df['event_time'] = pd.to_datetime(df['event_time'])

df['hour'] = df['event_time'].dt.hour

hourly = (
    df.groupby(['hour', 'event_type'])
      .size()
      .unstack(fill_value=0)
)

hourly['cart_rate'] = hourly['cart'] / hourly['view']
hourly['purchase_rate'] = hourly['purchase'] / hourly['cart']

hourly[['cart_rate', 'purchase_rate']]


# extract hour
df['hour'] = df['event_time'].dt.hour

# aggregate
hourly = (
    df.groupby(['hour', 'event_type'])
      .size()
      .unstack(fill_value=0)
)

hourly['cart_rate'] = hourly['cart'] / hourly['view']

# prepare for plotting
plot_df = hourly.reset_index()

# plot
fig = px.bar(
    plot_df,
    x='hour',
    y='cart_rate',
    title='Cart Rate by Hour of Day',
    text=(plot_df['cart_rate'] * 100).round(1)
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    width=900,
    height=400,
    template='plotly_white',
    xaxis_title='Hour of Day',
    yaxis_title='Cart Rate (%)',
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.update_yaxes(range=[0, plot_df['cart_rate'].max() * 1.2])

fig.show()

**Insight:** Time-based interventions (e.g., flash sales, time-of-day targeting) are unlikely to improve conversion—focus should remain on product/price-level optimization

---

## Key Drivers of Add-to-Cart Conversion

- **Product Type (Sub-category):** Significant variation across sub-categories indicates differing levels of user intent  
- **Brand:** Premium brands show lower cart rates, reflecting purchase hesitation at higher price points and comparison behavior  
- **Relative Price:** Users are sensitive to price differences within similar products, impacting add-to-cart decisions  

**Summary:** Conversion is primarily driven by intent, decision friction, and price comparison rather than lack of demand.

---

## Executive Summary

Add-to-cart conversion is the primary bottleneck, driven by product type, brand-level comparison behavior, and relative price sensitivity. Addressing these in high-volume categories like Electronics and Unknown could unlock ~15% revenue uplift.

## Final Recommendations

### 1. Focus on High-Impact Categories

- **Electronics:** High traffic with relatively low cart conversion  
- **Unknown Category:** Large volume with structural inefficiencies  

**These segments offer the highest potential return from conversion improvements**

---

### 2. Improve Catalog Structure (Unknown Category)

- A significant share of traffic falls under the “unknown” category with low conversion  
- This indicates weak product classification and poor discoverability  

**Action:**
- Improve category tagging and metadata quality  
- Ensure products are mapped to meaningful categories  

**Better categorization will improve product discovery and reduce browsing without conversion**

---

### 3. Reduce Decision Friction (Electronics)

- Electronics show lower cart rates due to higher comparison behavior  

**Action:**
- Introduce side-by-side product comparison  
- Highlight key differentiators (price, ratings, specs)  
- Surface top-rated or best-value products  

**Simplifying decision-making can significantly improve add-to-cart conversion**

---

### 4. Address Price Sensitivity

- Users are sensitive to price differences within similar products  

**Action:**
- Highlight value-for-money options  
- Use pricing cues (discounts, comparisons, bundles)  

**Improving price perception can reduce hesitation and increase conversions**




### **Final Takeaway:** Conversion optimization is primarily about reducing user hesitation rather than increasing demand.